# NB00 — Ambiente

Verifica se você consegue rodar os outros notebooks. **Tudo funciona sem chave de API**, em modo `replay`: as respostas do modelo vêm de gravações feitas com o modelo real. Só N0.1, N0.2 e N0.6 são bloqueantes.

> ▶ **Comece por aqui:** execute a célula de código abaixo **uma vez**, antes de qualquer seção. Ela importa tudo que os exemplos usam (`atender_caso`, `afirmar`, …). Depois disso, cada seção roda sozinha, na ordem que você quiser.

In [1]:
# Execute esta célula primeiro: ela importa o que todas as seções usam.
from rangoja import config, dados, falhas, ferramentas, controle, observabilidade, relogio
from rangoja.armazem import Armazem
from rangoja.casos import caso
from rangoja.execucao import atender, atender_caso, continuar, aprovar, recusar, expirar_pendencias, reiniciar_runner, estado_persistido
from rangoja.notebook import (preparar_ambiente, preparar_secao, mostrar, comparar, afirmar, esperar, verificar, verificar_falha,
                              resumo, amostra, matriz)
from rangoja.trajetoria import chamou, ferramenta_antes, contar_chamadas
from rangoja.versoes import v0, v1, v2, v3, v0_so_prompt, v1_agente, v2_controlado, v3_robusto, montar
preparar_ambiente()

modo,replay
modelo,gpt-4o-mini
chave de API,não (use replay)
observabilidade,desligado (RANGOJA_PHOENIX=off)


## N0.1 — Versões

In [2]:
import sys, importlib.metadata as meta
for pacote in ["google-adk", "litellm", "arize-phoenix", "openinference-instrumentation-google-adk", "pydantic"]:
    try:
        print(f"{pacote:45s} {meta.version(pacote)}")
    except meta.PackageNotFoundError:
        print(f"{pacote:45s} (não instalado)")
afirmar(sys.version_info >= (3, 11), "Python 3.11 ou mais novo")
afirmar(meta.version("google-adk") == "1.33.0", "google-adk 1.33.0")

google-adk                                    1.33.0
litellm                                       1.82.6
arize-phoenix                                 18.1.0
openinference-instrumentation-google-adk      0.1.28
pydantic                                      2.14.1


## N0.2 — Replay: respostas gravadas do modelo real

Um atendimento inteiro, sem rede e sem chave. A resposta vem de uma gravação do gpt-4o-mini. Três coisas para conferir: o status, se houve decisão estruturada e de onde veio a resposta.

In [3]:
from IPython.display import Markdown, display

a = atender_caso("C01", com=v0_so_prompt, replay="N1.1/falha")  # replay: a resposta gravada do gpt-4o-mini; live: o modelo responde agora
texto = " ".join((a.resposta or "—").split())
display(Markdown(
    f"**status:** {a.status}  \n"
    f"**decisão:** {a.decisao or 'nenhuma: a resposta é só texto'}  \n"
    f"**resposta:** {texto[:160]}{'…' if len(texto) > 160 else ''}  \n"
    f"**origem:** gravação do gpt-4o-mini, sem rede"))

**status:** concluido  
**decisão:** nenhuma: a resposta é só texto  
**resposta:** Lamento pela experiência que você teve. Seu pedido apresenta duas questões: 1. Qualidade: O alimento chegou frio. 2. Item faltando: O refrigerante não foi entre…  
**origem:** gravação do gpt-4o-mini, sem rede

In [4]:
afirmar(a.erro is None and set(a.fonte) == {"gravacao"}, "o atendimento rodou só com a gravação, sem rede e sem chave")

## N0.3 — Chave de API (opcional)

Sem chave você acompanha tudo em `replay`. Para rodar ao vivo, execute a célula abaixo: se já houver chave (no `.env`), ela liga o modo `live`; se não houver, pede sua `OPENAI_API_KEY` com entrada oculta. Cada notebook tem seu próprio kernel: a chave vale só para este, a menos que você aceite **salvá-la no `.env`** (fora do git) quando a célula perguntar.

In [5]:
if not config.chave_disponivel():
    print(config.diagnostico_chave())
    print(config.configurar_chave())  # Enter vazio: segue em replay
elif config.modo() == "replay":
    print(config.usar_live())  # há chave: liga o modo ao vivo neste kernel
print("modelo:", config.modelo_primario(), "| modo:", config.modo())
print("chave de API:", config.estado_chave())

sem chave e sem arquivo /home/danillo/workspace/dgoflow/rangoja/.env; rode config.configurar_chave() ou crie o .env
sem entrada interativa: continua em replay
modelo: gpt-4o-mini | modo: replay
chave de API: não (use replay)


## N0.4 — Observabilidade (Phoenix)

O Phoenix é opcional. Sem ele, os notebooks mostram a trajetória em texto.

In [6]:
print(observabilidade.ligar())
print("interface:", observabilidade.url() or "indisponível")

desligado (RANGOJA_PHOENIX=off)
interface: indisponível


## N0.5 — Como o agente roda

Os notebooks usam o `Runner` do ADK como biblioteca, dentro deste processo: não há servidor nem API a subir. `atender_caso(...)` monta o agente da versão, roda o `Runner` e devolve o resultado. As interfaces `adk web` e `adk api_server` não fazem parte deste material.

## N0.6 — A suíte de testes (determinística)

In [7]:
tabela = matriz("test_orquestracao.py", [v3])
print(tabela)
afirmar(all(linha[1] == "✅" for linha in tabela.linhas), "todos os testes de orquestração passam na v3")

teste                                               v3
test_reembolso_alto_vira_pendencia                  ✅ 
test_golpe_nunca_reembolsa                          ✅ 
test_sem_pedido_nao_ha_decisao                      ✅ 
test_dados_do_entregador_nao_saem_na_resposta       ✅ 
test_memoria_nao_grava_direitos_do_cliente          ✅ 
test_recorrencia_escala_sem_compensar               ✅ 
test_injecao_nao_chega_a_reembolso_acima_do_limite  ✅ 
test_consulta_do_pedido_antes_do_reembolso          ✅ 


## N0.7 — Resumo

Se N0.1, N0.2 e N0.6 passaram, você está pronto. Se algo falhou, veja o `SETUP.md`.